# code 1

In [1]:
import numpy as np
import pandas as pd
from scipy.stats import chi2


# ============================================================
# Kendall's coefficient of concordance W
# ============================================================

def kendalls_w_from_rank_matrix(rank_matrix):
    """
    Compute Kendall's coefficient of concordance W.

    Parameters
    ----------
    rank_matrix : array-like, shape (m, n)
        Rows are judges.
        Columns are ranked items.
        Entries are ranks.
        Here:
            m = number of judges
            n = number of methods

    Returns
    -------
    W : float
        Kendall's coefficient of concordance.
    chi2_stat : float
        Chi-square test statistic m(n - 1)W.
    df : int
        Degrees of freedom n - 1.
    p_value : float
        Approximate p-value from chi-square distribution.
    rank_sums : ndarray
        Sum of ranks for each item.
    """

    R = np.asarray(rank_matrix, dtype=float)

    if R.ndim != 2:
        raise ValueError("rank_matrix must be a 2D array.")

    m, n = R.shape

    rank_sums = R.sum(axis=0)
    mean_rank_sum = m * (n + 1) / 2.0

    S = np.sum((rank_sums - mean_rank_sum) ** 2)

    W = 12.0 * S / (m**2 * (n**3 - n))

    chi2_stat = m * (n - 1) * W
    df = n - 1
    p_value = chi2.sf(chi2_stat, df)

    return W, chi2_stat, df, p_value, rank_sums


def scores_to_ranks(scores, higher_is_better=True):
    """
    Convert scores to ranks.

    Rank 1 means best method.

    Parameters
    ----------
    scores : pandas.Series
        Scores indexed by method.
    higher_is_better : bool
        If True, larger score receives better rank.
        If False, smaller score receives better rank.

    Returns
    -------
    pandas.Series
        Ranks indexed by method.
    """

    return scores.rank(
        ascending=not higher_is_better,
        method="average"
    )


# ============================================================
# Input data from the manuscript table
# ============================================================

methods = ["CAKR", "NVM", "FFP-JS", "FFP-KL", "MKS", "FPS"]

metrics = ["ACC", "BA", "F1", "Recall", "Precision"]

data = {
    "NCBI 2020": {
        "CAKR":   [0.913, 0.887, 0.892, 0.887, 0.915],
        "NVM":    [0.847, 0.807, 0.809, 0.807, 0.840],
        "FFP-JS": [0.821, 0.790, 0.781, 0.790, 0.814],
        "FFP-KL": [0.819, 0.789, 0.780, 0.789, 0.814],
        "MKS":    [0.713, 0.644, 0.622, 0.644, 0.668],
        "FPS":    [0.714, 0.637, 0.633, 0.637, 0.665],
    },

    "NCBI 2022": {
        "CAKR":   [0.902, 0.820, 0.824, 0.819, 0.859],
        "NVM":    [0.852, 0.747, 0.750, 0.747, 0.791],
        "FFP-JS": [0.830, 0.740, 0.733, 0.740, 0.769],
        "FFP-KL": [0.832, 0.743, 0.735, 0.743, 0.771],
        "MKS":    [0.724, 0.593, 0.577, 0.593, 0.617],
        "FPS":    [0.723, 0.588, 0.580, 0.588, 0.603],
    },

    "NCBI 2024": {
        "CAKR":   [0.876, 0.792, 0.804, 0.792, 0.853],
        "NVM":    [0.814, 0.729, 0.738, 0.729, 0.789],
        "FFP-JS": [0.796, 0.724, 0.712, 0.724, 0.744],
        "FFP-KL": [0.796, 0.727, 0.714, 0.727, 0.747],
        "MKS":    [0.633, 0.589, 0.554, 0.589, 0.573],
        "FPS":    [0.660, 0.561, 0.560, 0.561, 0.593],
    },

    "NCBI 2024 All": {
        "CAKR":   [0.876, 0.794, 0.807, 0.794, 0.853],
        "NVM":    [0.809, 0.729, 0.738, 0.729, 0.788],
        "FFP-JS": [0.799, 0.721, 0.711, 0.721, 0.745],
        "FFP-KL": [0.799, 0.724, 0.712, 0.724, 0.745],
        "MKS":    [0.638, 0.589, 0.555, 0.589, 0.575],
        "FPS":    [0.651, 0.561, 0.559, 0.561, 0.591],
    },
}


# Convert to one dataframe:
# rows = dataset/method, columns = metrics

rows = []

for dataset_name, method_dict in data.items():
    for method_name, values in method_dict.items():
        row = {
            "Dataset": dataset_name,
            "Method": method_name,
        }
        row.update(dict(zip(metrics, values)))
        rows.append(row)

df = pd.DataFrame(rows)

print("\nInput performance table:")
print(df)


# ============================================================
# Analysis I:
# Agreement of method rankings across metrics within each dataset
# ============================================================

type_I_results = []
type_I_rank_tables = {}

for dataset_name in data.keys():
    sub = df[df["Dataset"] == dataset_name].set_index("Method")

    rank_rows = []

    for metric in metrics:
        ranks = scores_to_ranks(sub[metric], higher_is_better=True)
        rank_rows.append(ranks.loc[methods].values)

    rank_matrix = np.vstack(rank_rows)

    W, chi2_stat, df_chi, p_value, rank_sums = kendalls_w_from_rank_matrix(rank_matrix)

    type_I_results.append({
        "Dataset": dataset_name,
        "Kendalls_W": W,
        "Chi_square": chi2_stat,
        "df": df_chi,
        "p_value": p_value,
    })

    rank_table = pd.DataFrame(rank_matrix, index=metrics, columns=methods)
    type_I_rank_tables[dataset_name] = rank_table

type_I_results_df = pd.DataFrame(type_I_results)

print("\nKendall's W: rankings across metrics within each dataset")
print(type_I_results_df)


# ============================================================
# Analysis II:
# Agreement of method rankings across dataset versions for each metric
# ============================================================

type_II_results = []
type_II_rank_tables = {}

for metric in metrics:
    rank_rows = []

    for dataset_name in data.keys():
        sub = df[df["Dataset"] == dataset_name].set_index("Method")
        ranks = scores_to_ranks(sub[metric], higher_is_better=True)
        rank_rows.append(ranks.loc[methods].values)

    rank_matrix = np.vstack(rank_rows)

    W, chi2_stat, df_chi, p_value, rank_sums = kendalls_w_from_rank_matrix(rank_matrix)

    type_II_results.append({
        "Metric": metric,
        "Kendalls_W": W,
        "Chi_square": chi2_stat,
        "df": df_chi,
        "p_value": p_value,
    })

    rank_table = pd.DataFrame(rank_matrix, index=list(data.keys()), columns=methods)
    type_II_rank_tables[metric] = rank_table

type_II_results_df = pd.DataFrame(type_II_results)

print("\nKendall's W: rankings across dataset versions for each metric")
print(type_II_results_df)


# ============================================================
# Save outputs
# ============================================================

df.to_csv("kendalls_w_input_performance_table.csv", index=False)
type_I_results_df.to_csv("kendalls_w_across_metrics_within_dataset.csv", index=False)
type_II_results_df.to_csv("kendalls_w_across_datasets_within_metric.csv", index=False)

with pd.ExcelWriter("kendalls_w_rank_tables.xlsx") as writer:
    df.to_excel(writer, sheet_name="input_scores", index=False)
    type_I_results_df.to_excel(writer, sheet_name="W_across_metrics", index=False)
    type_II_results_df.to_excel(writer, sheet_name="W_across_datasets", index=False)

    for dataset_name, rank_table in type_I_rank_tables.items():
        safe_name = dataset_name.replace(" ", "_")[:31]
        rank_table.to_excel(writer, sheet_name=f"I_{safe_name}"[:31])

    for metric, rank_table in type_II_rank_tables.items():
        rank_table.to_excel(writer, sheet_name=f"II_{metric}"[:31])


print("\nSaved files:")
print("  kendalls_w_input_performance_table.csv")
print("  kendalls_w_across_metrics_within_dataset.csv")
print("  kendalls_w_across_datasets_within_metric.csv")
print("  kendalls_w_rank_tables.xlsx")


Input performance table:
          Dataset  Method    ACC     BA     F1  Recall  Precision
0       NCBI 2020    CAKR  0.913  0.887  0.892   0.887      0.915
1       NCBI 2020     NVM  0.847  0.807  0.809   0.807      0.840
2       NCBI 2020  FFP-JS  0.821  0.790  0.781   0.790      0.814
3       NCBI 2020  FFP-KL  0.819  0.789  0.780   0.789      0.814
4       NCBI 2020     MKS  0.713  0.644  0.622   0.644      0.668
5       NCBI 2020     FPS  0.714  0.637  0.633   0.637      0.665
6       NCBI 2022    CAKR  0.902  0.820  0.824   0.819      0.859
7       NCBI 2022     NVM  0.852  0.747  0.750   0.747      0.791
8       NCBI 2022  FFP-JS  0.830  0.740  0.733   0.740      0.769
9       NCBI 2022  FFP-KL  0.832  0.743  0.735   0.743      0.771
10      NCBI 2022     MKS  0.724  0.593  0.577   0.593      0.617
11      NCBI 2022     FPS  0.723  0.588  0.580   0.588      0.603
12      NCBI 2024    CAKR  0.876  0.792  0.804   0.792      0.853
13      NCBI 2024     NVM  0.814  0.729  0.738   0